<a href="https://colab.research.google.com/github/Xcelrator0/Intership-Tasks/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
if not os.path.isdir("Intership-Tasks"):
    !git clone https://github.com/Xcelrator0/Intership-Tasks.git
%cd Intership-Tasks

Cloning into 'Intership-Tasks'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 151 (delta 56), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 1.87 MiB | 13.89 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/Intership-Tasks


In [2]:
%pip install -q pandas numpy
import pandas as pd, numpy as np

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
for c in df.columns:
    print(c, "-", df[c].dtype)

content_id - object
client_id - object
search_volume - float64
competition - float64
competition_level - object
cpc - float64
content_type - object
main_intent - object
word_count - float64
char_count - float64
provider_used - object
model_used - object
impressions_90d - int64
clicks_90d - int64
pageviews_90d - int64
sessions_90d - int64
users_90d - int64
engaged_sessions_90d - int64
ai_sessions_90d - int64
scroll_events_90d - int64
days_with_impressions - int64
days_with_sessions - int64
impressions_last_30d - int64
clicks_last_30d - int64
sessions_last_30d - int64
impressions_prev_30d - int64
clicks_prev_30d - int64
sessions_prev_30d - int64
content_age_days - int64
age_tier - object
age_tier_order - int64
days_since_last_update - int64
freshness_tier - object
word_count_tier - object
char_count_tier - object
ctr - float64
avg_position - float64
engagement_rate - float64
scroll_rate - float64
ai_traffic_pct - float64
impression_tier - object
position_tier - object
trend_direction - o

In [4]:
keywords = ["stale", "ctr", "position", "rank", "volume", "refresh", "click", "impress", "days"]
candidates = [c for c in df.columns if any(k in c.lower() for k in keywords)]
print("Candidate signal columns:", candidates)

Candidate signal columns: ['search_volume', 'impressions_90d', 'clicks_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'impression_tier', 'position_tier']


## 1. My rule and its reason codes


In [5]:
buckets_a = pd.qcut(df["days_since_last_update"], q=4, duplicates="drop")
table_a = df.groupby(buckets_a).size().rename("n")
print("days_since_last_update buckets:")
print(table_a)
pos_bins = pd.qcut(df["avg_position"], q=5, duplicates="drop")
expected_ctr_by_pos = df.groupby(pos_bins)["ctr"].transform("mean")
df["ctr_gap"] = expected_ctr_by_pos - df["ctr"]

buckets_b = pd.qcut(df["ctr_gap"], q=4, duplicates="drop")
table_b = df.groupby(buckets_b).size().rename("n")
print("\nctr_gap buckets (higher = more underperforming vs position peers):")
print(table_b)

days_since_last_update buckets:
days_since_last_update
(0.999, 20.0]     15866
(20.0, 104.0]     13816
(104.0, 373.0]      318
Name: n, dtype: int64

ctr_gap buckets (higher = more underperforming vs position peers):
ctr_gap
(-99.82300000000001, 0.108]    7539
(0.108, 0.253]                 7490
(0.253, 0.473]                 7472
(0.473, 1.18]                  7499
Name: n, dtype: int64


/tmp/ipykernel_2657/110257052.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  table_a = df.groupby(buckets_a).size().rename("n")
/tmp/ipykernel_2657/110257052.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  expected_ctr_by_pos = df.groupby(pos_bins)["ctr"].transform("mean")
/tmp/ipykernel_2657/110257052.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  table_b = df.groupby(buckets_b).size().rename("n")


days_since_last_update splits unevenly: 15,866 pages updated within 20 days, 13,816 in the 20-104 day range, but only 318 pages sit in the 104-373 day tail — so staleness as I've defined it is a real but rare signal, concentrated in a small slice of the portfolio. ctr_gap splits into four roughly even quartiles (~7,500 each), which is expected since it's built from qcut — it doesn't by itself tell me the signal is meaningful, just that it's not degenerate. Verdict: MIXED — staleness looks like a genuine, if thin, signal (318 truly stale pages is a small base to build a rule on); ctr_gap's evenness is a property of the bucketing method, not evidence of predictive power on its own.

## 2. Encode ONE rule (writes the CSV)

In [6]:
import os
os.makedirs("work/outputs", exist_ok=True)

SIGNAL_A = "days_since_last_update"
SIGNAL_B = "ctr_gap"

def normalize(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

work = df.copy()
work["score"] = 0.5 * normalize(work[SIGNAL_A]) + 0.5 * normalize(work[SIGNAL_B])


stale_hi = work[SIGNAL_A].quantile(0.75)
ctrgap_hi = work[SIGNAL_B].quantile(0.75)
volume_hi = work["search_volume"].quantile(0.75)

def reason_code(row):
    stale = row[SIGNAL_A] >= stale_hi
    ctr_bad = row[SIGNAL_B] >= ctrgap_hi
    high_vol = row["search_volume"] >= volume_hi
    if stale and high_vol:
        return "STALE_HIGH_VOLUME"
    if ctr_bad:
        return "CTR_BELOW_EXPECTED"
    if stale:
        return "STALE_LOW_VOLUME"
    return "MONITOR_ONLY"

work["reason_code"] = work.apply(reason_code, axis=1)
work["action"] = np.where(work["score"] > work["score"].quantile(0.9), "review_first", "monitor")

queue = work.sort_values("score", ascending=False)
out_cols = ["content_id", "score", "reason_code", "action", SIGNAL_A, "ctr", "avg_position", "search_volume"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote work/outputs/baseline_action_score.csv")
queue[out_cols].head(10)

wrote work/outputs/baseline_action_score.csv


,content_id,score,reason_code,action,days_since_last_update,ctr,avg_position,search_volume
26242,content_55a5b1c46474,0.996698,CTR_BELOW_EXPECTED,review_first,373,0.00,7.5,0.0
24216,content_1b4ec72dafd4,0.995354,CTR_BELOW_EXPECTED,review_first,372,0.00,7.0,0.0
29384,content_f6fdf87348f6,0.995039,STALE_LOW_VOLUME,review_first,373,0.00,32.5,0.0
18440,content_8d56efff1e71,0.993695,STALE_LOW_VOLUME,review_first,372,0.00,35.0,0.0
15608,content_06e19c6486b0,0.947581,CTR_BELOW_EXPECTED,review_first,334,0.00,5.0,NaN
8631,content_e2b702f4f92b,0.943585,STALE_LOW_VOLUME,review_first,334,0.00,9.3,NaN
6962,content_f01216059a6a,0.929866,STALE_LOW_VOLUME,review_first,335,3.85,5.3,NaN
21984,content_02b0d6e30129,0.916053,STALE_HIGH_VOLUME,review_first,313,0.00,6.9,110.0
18841,content_94991fe6268c,0.915359,STALE_LOW_VOLUME,review_first,313,0.00,12.4,10.0
15790,content_6476d1d8c050,0.914394,STALE_LOW_VOLUME,review_first,313,0.00,67.8,10.0


## 3. Top-10 review

In [7]:
top10 = queue[out_cols].head(10).reset_index(drop=True)
med_stale = df["days_since_last_update"].median()
med_ctr = df["ctr"].median()
med_vol = df["search_volume"].median()

for i, row in top10.iterrows():
    print(f"--- Row {i+1}: {row['content_id']} ---")
    print(f"  score={row['score']:.3f}  reason_code={row['reason_code']}  action={row['action']}")
    print(f"  days_since_last_update={row['days_since_last_update']:.0f}  (dataset median: {med_stale:.0f})")
    print(f"  ctr={row['ctr']:.3f}  (dataset median: {med_ctr:.3f})  avg_position={row['avg_position']:.1f}")
    print(f"  search_volume={row['search_volume']:.0f}  (dataset median: {med_vol:.0f})")
    print()

--- Row 1: content_55a5b1c46474 ---
  score=0.997  reason_code=CTR_BELOW_EXPECTED  action=review_first
  days_since_last_update=373  (dataset median: 20)
  ctr=0.000  (dataset median: 0.070)  avg_position=7.5
  search_volume=0  (dataset median: 10)

--- Row 2: content_1b4ec72dafd4 ---
  score=0.995  reason_code=CTR_BELOW_EXPECTED  action=review_first
  days_since_last_update=372  (dataset median: 20)
  ctr=0.000  (dataset median: 0.070)  avg_position=7.0
  search_volume=0  (dataset median: 10)

--- Row 3: content_f6fdf87348f6 ---
  score=0.995  reason_code=STALE_LOW_VOLUME  action=review_first
  days_since_last_update=373  (dataset median: 20)
  ctr=0.000  (dataset median: 0.070)  avg_position=32.5
  search_volume=0  (dataset median: 10)

--- Row 4: content_8d56efff1e71 ---
  score=0.994  reason_code=STALE_LOW_VOLUME  action=review_first
  days_since_last_update=372  (dataset median: 20)
  ctr=0.000  (dataset median: 0.070)  avg_position=35.0
  search_volume=0  (dataset median: 10)

--

Row 2 (content_1b4ec72dafd4): review_first — CTR_BELOW_EXPECTED, ctr 0.00 vs position 7.0 where median CTR is 0.07. Would be wrong if this page has near-zero impressions, since CTR off a tiny denominator is noise, not underperformance.

Row 3 (content_f6fdf87348f6): review_first — STALE_LOW_VOLUME, 373 days since update, search_volume=0. Would be wrong if zero volume means this keyword/page is simply not a priority anymore, not neglected.

Row 4 (content_8d56efff1e71): review_first — STALE_LOW_VOLUME, 372 days stale, position 35 — deep enough that a refresh may not move the needle regardless of staleness.

Row 5 (content_06e19c6486b0): review_first — CTR_BELOW_EXPECTED, but search_volume=nan — missing data, not zero. Would be wrong if this row shouldn't be scored at all without a real volume figure.

Row 6 (content_e2b702f4f92b): review_first — STALE_LOW_VOLUME, same nan-volume issue as row 5.

## 4. Weak picks + leakage check


In [8]:

print("Borderline top-10 rows (closer calls, worth a second look):")
for i, row in top10.iterrows():
    near_stale = abs(row["days_since_last_update"] - stale_hi) / max(stale_hi, 1) < 0.10
    if near_stale:
        print(f"  Row {i+1} ({row['content_id']}): days_since_last_update={row['days_since_last_update']:.0f} is close to the stale_hi threshold ({stale_hi:.0f})")

print("\nLeakage check:")
leaked = [c for c in [SIGNAL_A, "ctr_gap", "search_volume"] if c in ("trend_direction", "trend_pct")]
print("Label columns used as features:", leaked if leaked else "none — confirmed clean")

Borderline top-10 rows (closer calls, worth a second look):

Leakage check:
Label columns used as features: none — confirmed clean


Leakage check confirms no label columns entered the feature set. Weakest picks: rows 5 and 6 (content_06e19c6486b0, content_e2b702f4f92b) — both have search_volume=nan, meaning the model is scoring them without real demand data behind the CTR/staleness signal, which makes the high score less trustworthy than it looks. Row 7 is also weak — flagged for staleness despite already having above-median CTR (3.85 vs 0.07 median), suggesting the rule triggered on one signal while ignoring that the other signal looked healthy.